# Fire Spread Celular Automata Model Calibration
_Author: Paulo Medina_

In [1]:
# Import libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import data_preparation_functions

# Get Fire test date and centroid

In [2]:
# Import data
df_catalog = pd.read_csv("../01_Data/05_Spread_Covariates/fire_event_catalog.csv")
fire_event = df_catalog[df_catalog["event_id"] == 5072].iloc[0]
print(f"Fire Event {fire_event['event_id']}\nStart date: {fire_event['start_date']}\nEnd date: {fire_event['end_date']}")

Fire Event 5072
Start date: 2019-10-28
End date: 2019-11-02


In [3]:
fire_points_gdf = gpd.read_file("../01_Data/06_Wildfire_Clusters/Fire_clusters_chaco.shp")
fire_points_gdf.head()

,SOURCE_ID,CLUSTER_ID,COLOR_ID,ACQ_DATE,START_TIME,END_TIME,MEAN_TIME,TIME_EXAGG,geometry
0,1,-1,-1,2020-08-29,2000-11-01,2023-12-30,2012-09-07,42800.567443,POINT (-57.6865 -25.28)
1,2,-1,-1,2010-02-12,2000-11-01,2023-12-30,2012-09-07,20037.829531,POINT (-57.72 -25.2474)
2,3,-1,-1,2006-08-07,2000-11-01,2023-12-30,2012-09-07,12442.369074,POINT (-57.7157 -25.2436)
3,4,-1,-1,2017-07-25,2000-11-01,2023-12-30,2012-09-07,36115.380069,POINT (-57.6813 -25.2641)
4,5,-1,-1,2003-07-12,2000-11-01,2023-12-30,2012-09-07,5810.379477,POINT (-57.6917 -25.2638)


In [4]:
ca_data = data_preparation_functions.prepare_ca_inputs(5072, 
                                             r"..\01_Data\07_SRTM\SRTM_Paraguay_Chaco.tif", 
                                             r"..\01_Data\03_MapBiomas\2019_coverage_lclu_25-1-1_3b6f05ac-90e3-4fca-9797-3e0ce1b05225.tif", 
                                             r"..\01_Data\05_Spread_Covariates\02_Weather\era5_timeseries_5072.csv", 
                                             fire_points_gdf)

Preparing CA inputs for event 5072...
  Loading elevation...
  Calculating slope and aspect...
  Loading land use...
  Snapping land use to elevation grid...
  Mapping to fuel types...
  Loading weather time series...
  Creating ignition raster...
  Grid shape: (22256, 20369)
  Grid CRS: EPSG:4326
  Ignition time: 2019-10-28 00:00:00
Done.


np.float64(6.281193277941991)

# Preparing Fuel Data Layers

Land Cover data will be mapped to fuel types and their associated K value derived from _Gomes et al. 2020_. 

- 1: 0.04,  # Forest
- 2: 0.96,  # Savanna/Cerrado
- 3: 1.00,  # Grassland (reference)
- 4: 0.60,  # Cropland (estimated - needs justification)
- 5: 0.00   # Non-combustible

**Note on Fuel Type 4 (Cropland) Ks = 0.6**: 
- This is **estimated** based on the qualitative expectation that cropland spreads slower than grassland but faster than forest
- Gomes et al. (2020) didn't have sufficient cropland fire data
- You should cite this as a "conservative intermediate value" in your methods
- Can be refined during calibration if needed



# Ingest rasters

All rasters will be ingested using `rasterio`.

They will be re-sampled

In [5]:
arr = np.array([[1, 2, 3], [4, 5, 6]])

for (i, j), value in np.ndenumerate(arr):
    print(f"Index: ({i}, {j}), Value: {value}")

Index: (0, 0), Value: 1
Index: (0, 1), Value: 2
Index: (0, 2), Value: 3
Index: (1, 0), Value: 4
Index: (1, 1), Value: 5
Index: (1, 2), Value: 6
